# Review capacity, thresholds, and friction

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
assert (REPORTS / "run_manifest.json").exists(), "Run python -m fraudgraph.pipeline --download first"
manifest = json.loads((REPORTS / "run_manifest.json").read_text())
print("Evidence run:", manifest["run_id"])
def figure(name):
    display(Image(filename=str(REPORTS / "figures" / name)))


Evidence run: 20260923T163139886696Z


In [2]:
policy = json.loads((REPORTS / "decision_policy.json").read_text())
display(pd.Series(policy))
display(pd.read_csv(REPORTS / "risk_band_labels.csv"))

selected_model                                              named_xgboost_graph
low_review_boundary                                                    0.807129
review_high_boundary                                                   0.808545
high_enabled                                                               True
high_precision_target                                                       0.9
review_fraction_validation                                                 0.05
high_min_labeled_cases                                                       20
test_band_counts                      {'LOW': 44963, 'HIGH': 1679, 'REVIEW': 5}
note                          High precision is empirical on labeled validat...
dtype: object

,risk_band,1,2,3
0,HIGH,291,76,1312
1,LOW,345,10472,34146
2,REVIEW,0,0,5


In [3]:
capacity = pd.read_csv(REPORTS / "review_capacity.csv")
chosen = capacity[(capacity.split == "test") & (capacity.experiment_id == policy["selected_model"])]
display(chosen)

,scope,capacity_fraction,reviewed,known_illicit,known_licit,unknown,precision_at_k_known,recall_at_k_known,precision_lower_bound_all,precision_upper_bound_all,experiment_id,split
70,labeled_only,0.01,117,48,69,0,0.410256,0.075472,0.410256,0.410256,named_xgboost_graph,test
71,labeled_only,0.02,229,93,136,0,0.406114,0.146226,0.406114,0.406114,named_xgboost_graph,test
72,labeled_only,0.05,564,215,349,0,0.381206,0.338050,0.381206,0.381206,named_xgboost_graph,test
73,labeled_only,0.10,1125,329,796,0,0.292444,0.517296,0.292444,0.292444,named_xgboost_graph,test
74,labeled_only,0.20,2243,394,1849,0,0.175658,0.619497,0.175658,0.175658,named_xgboost_graph,test
75,all_transactions,0.01,471,83,18,370,0.821782,0.130503,0.176221,0.961783,named_xgboost_graph,test
76,all_transactions,0.02,938,142,52,744,0.731959,0.223270,0.151386,0.944563,named_xgboost_graph,test
77,all_transactions,0.05,2337,273,140,1924,0.661017,0.429245,0.116816,0.940094,named_xgboost_graph,test
78,all_transactions,0.10,4670,324,530,3816,0.379391,0.509434,0.069379,0.886510,named_xgboost_graph,test
79,all_transactions,0.20,9333,389,1427,7517,0.214207,0.611635,0.041680,0.847102,named_xgboost_graph,test


The fixed risk bands and top-K capacity policies are different. A fixed score threshold can exceed or undershoot staffing capacity as traffic changes. Top-K caps workload but does not enforce a minimum confidence. Unknown cases consume analyst time, with unknown investigative yield.

In [4]:
display(pd.read_csv(REPORTS / "sql_4.csv"))
display(Markdown((REPORTS / "DECISION_FRAMEWORK.md").read_text()))

,risk_band,volume,known_illicit,known_licit,unknown
0,HIGH,1679,291,76,1312
1,LOW,44963,345,10472,34146
2,REVIEW,5,0,0,5


# Risk triage and customer friction

These are risk scores for an analyst workflow, not calibrated fraud probabilities or automatic grounds to block a customer.

1. Select the primary model with highest validation AP, before examining test performance.
2. Set the LOW/REVIEW boundary to the 95th percentile of scores on **all** validation transactions. A 5% review fraction is an explicit illustrative staffing scenario, not an estimated business optimum.
3. Among score thresholds at or above that boundary, set the REVIEW/HIGH boundary to the lowest threshold achieving at least 90% observed precision on at least 20 labeled validation transactions. If no threshold qualifies, disable HIGH. The empirical target is not a confidence guarantee and excludes unknown outcomes.
4. Freeze both boundaries and report subsequent test volumes. Distribution shift and ties mean the 5% validation scenario is not a hard test-volume cap.

LOW means no automatic escalation by this model; it does not certify innocence. REVIEW means routine manual investigation. HIGH means prioritized human investigation; automatic blocking is not justified by this study. Known licit cases in these queues are a measurable proxy for unnecessary review and possible friction, not a count of actual customer harm. Transaction-level data cannot quantify unique customers or their costs.

`review_capacity.csv` separately evaluates an enforceable ranking policy: take the top ceil(fraction Ã— bucket volume) per time step, breaking score ties by transaction ID, at 1%, 2%, 5%, 10%, and 20%. The labeled-only benchmark supplies conventional Precision@K and Recall@K. The all-traffic view charges unknown cases against analyst capacity and reports known illicit capture, known licit review, unknown volume, and lower/upper precision bounds. It cannot estimate population recall because unknown illicit cases have no ground truth.

Do not silently conflate these two policies. Fixed score bands provide consistent evidence thresholds but variable workload; per-bucket top-K enforces workload but changes the effective threshold as traffic changes. Real deployment requires analyst cost, loss severity, label maturation, calibrated scores, fairness checks, and prospective validation.


HIGH means priority human review, not automatic blocking. A narrow REVIEW interval is a measured outcome of this illustrative policy, not a reason to tune thresholds on test. The large validation-to-test performance decline means prospective validation is required before deployment.